In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 5 — O MAPA DO TERRENO: EDA PARA CLASSIFICAÇÃO

> "Um bom general não entra em batalha sem um mapa detalhado do terreno. Para nós, esse mapa é a Análise Exploratória de Dados."
> — Sun Tzu, adaptado para a Ciência de Dados

Com o conjunto de teste trancado no cofre, eu finalmente me senti livre para explorar. O `X_train` era agora meu mundo — um microcosmo do problema. No OncoClassify 1.0, minha análise foi apressada e feita sobre o dataset completo. Foi como olhar um mapa-múndi quando eu precisava de um mapa topográfico de uma única região.

Lembrei de Helena. O que, nos dados dela, tornou o diagnóstico tão ambíguo? Para evitar que isso se repetisse, eu precisava entender a geografia de cada *feature*: onde as classes se separam com clareza, e onde se confundem. Não era a caça a uma "*feature* mágica", mas o reconhecimento paciente do terreno.

## EDA Para Classificação: Buscando a Separabilidade

Em problemas de classificação, o objetivo da EDA é investigar a **separabilidade das classes**. Ao contrário da regressão, onde buscamos tendências, aqui buscamos **diferenças nas distribuições**: existe alguma *feature* que, sozinha, separe bem malignos de benignos? Onde as classes se sobrepõem?

Toda a análise usa **apenas** `X_train`/`y_train` — jamais o teste, sob pena de *data leakage*. Nossas ferramentas visuais serão o **box plot** (mediana, quartis, *outliers*) e o **violin plot** (a forma completa da distribuição), lado a lado por classe.

#### Código 5.1: Distribuições por Classe


In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
from oncoclassify_utils import (X_train, y_train, color_map,
                                apply_hatches_boxplot, apply_hatches_violinplot)

df_tr = X_train.copy()
df_tr["diagnosis_label"] = y_train.map({0: "Maligno", 1: "Benigno"}).values
ordem = ["Benigno", "Maligno"]; pal = [color_map[c] for c in ordem]

for feature in ["mean radius", "mean texture", "worst concave points"]:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    sns.boxplot(data=df_tr, x="diagnosis_label", y=feature, hue="diagnosis_label",
                order=ordem, palette=pal, legend=False, ax=ax1)
    apply_hatches_boxplot(ax1, ordem); ax1.set_title(f"Box Plot: {feature}")
    sns.violinplot(data=df_tr, x="diagnosis_label", y=feature, hue="diagnosis_label",
                   order=ordem, palette=pal, legend=False, ax=ax2)
    apply_hatches_violinplot(ax2, ordem); ax2.set_title(f"Violin Plot: {feature}")
    plt.suptitle(f"Análise da Feature: {feature}", fontsize=15)
    plt.tight_layout(); plt.show()

![mean radius](media/figura_05_1.png)

![mean texture](media/figura_05_2.png)

![worst concave points](media/figura_05_3.png)

Os gráficos revelam três níveis de separabilidade:

**`mean radius`** — separação clara: mediana benigna ≈ 12,2 contra maligna ≈ 17,2. Forte preditor, com alguma sobreposição.

**`mean texture`** — separação fraca: medianas próximas (17,2 vs 21,3) e muita sobreposição. Sozinha, informa pouco.

**`worst concave points`** — separação excelente: mediana benigna ≈ 0,074 contra maligna ≈ 0,183, com sobreposição mínima. Visualmente, uma das *features* mais poderosas do dataset.

*Features* de tamanho (`radius`) e de irregularidade de forma (`concave points`) despontam como cruciais.

#### Código 5.2: Correlação Entre Features


In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
from oncoclassify_utils import X_train

plt.figure(figsize=(14, 12))
sns.heatmap(X_train.corr(), cmap="coolwarm", center=0, annot=False)
plt.title("Matriz de Correlação das Features (Treino)", fontsize=15)
plt.tight_layout(); plt.show()

![Matriz de correlação](media/figura_05_4.png)

O mapa de calor mostra blocos quentes de forte correlação: `mean radius`, `mean perimeter` e `mean area` são quase perfeitamente correlacionadas (perímetro e área derivam do raio), e muitas *features* "mean" se repetem nas "worst". Isso é **multicolinearidade**: informação redundante. Sugere que talvez não precisemos das 30 *features* — a **seleção** (Capítulo 7) poderá enxugar o conjunto sem perder poder preditivo.

> **📌 CHECKLIST DO CAPÍTULO 5**
>
> ✅ Entende que a EDA para classificação busca **separabilidade das classes**.
>
> ✅ Sabe ler box plots e violin plots e identificar uma *feature* discriminativa (`worst concave points`) de uma fraca (`mean texture`).
>
> ✅ Fez toda a análise **apenas no treino**, evitando *leakage*.
>
> ✅ Identificou **multicolinearidade** forte entre as *features* de tamanho.
>
> **⚠️ CRÍTICO:** As conclusões daqui (ex.: `worst concave points` é ótima) são *hipóteses* que guiam o desenvolvimento — só serão validadas de forma justa no fim, com o teste.

O mapa do terreno tomava forma. As visualizações transformaram a tabela de números em uma paisagem de montanhas e vales: `worst concave points` era uma rodovia larga e bem sinalizada; `mean texture`, uma trilha sinuosa. Vi também a redundância — muitas estradas correndo paralelas ao mesmo destino.

Antes de podar, porém, eu quis enriquecer. A engenharia de *features* não é só usar o que nos dão, mas criar novas perspectivas combinando o que já existe. Era hora de passar de cartógrafo a escultor.
